# Stage 10 — CPU input and provenance audit

Private preparation for the frozen Kirill V1 361634630 recipe, whose author displays 0.639 official public balanced accuracy. Our measured official best remains S03 at 0.620.

This notebook reads input inventory, metadata, archive hashes and safe array headers. It executes no upstream model/source, pickle, joblib, Torch checkpoint, custom model code, fitting or inference. It exports custom Python text separately for private static review. A CPU pass cannot establish GPU/runtime validity. The source's ordinary three-row preview uses constant labels and is format-only.


In [ ]:
from pathlib import Path
import json,hashlib,gzip,base64
AUDIT_SOURCE = '"""CPU-only input audit for Kirill source V1 361634630; never run its source.\n\nUses stdlib, NumPy and pandas. No Torch, Transformers, joblib, pickle, model\nimports, custom code, fitting, inference or network calls are permitted here.\nCustom Python/config text is copied only into a separate PRIVATE review export.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport glob\nimport hashlib\nimport json\nimport re\nimport struct\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nSOURCE_VERSION = 361634630\nUPSTREAM_SHA = "1d302075c9b5f620db265a147cc11138e786a11ea754c262a33b9223879905de"\nCOMP_STACK_SHA = "c09e4dcb31fd6cc4c34578c053b22dc425eb8a3a341a1db579e1fd733598c792"\nTRAIN_SHA = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nLABELS = [\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\',\n          \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\',\n          \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\', \'claude-fable-5\',\n          \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\', \'kimi-k3\']\nSORTED_LABELS = sorted(LABELS)\nEXTRA_SOURCES = [(\'**/ntl_extra_passages_claude.csv\', \'model\', None), (\'**/gen_passages.csv\', \'model\', None),\n                 (\'**/ntl_generations.csv\', \'label\', \'finish_reason\'), (\'**/ntl_proxy_generations.csv\', \'label\', None),\n                 (\'**/ntl_generated*.csv\', \'model\', \'finish\'), (\'**/passages_clean.csv\', \'model\', None)]\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    path = Path(path)\n    if path.stat().st_size > 4 * 1024 * 1024:\n        raise ValueError(\'JSON metadata exceeds bounded reader: \' + path.name)\n    return json.loads(path.read_text(encoding=\'utf-8\'))\n\n\ndef safe_relative(name):\n    path = Path(str(name))\n    if path.is_absolute() or \'..\' in path.parts or not path.parts or \'\\x00\' in str(name):\n        raise ValueError(\'Unsafe relative asset path\')\n    return path\n\n\ndef checked_labels(value, ordered=None):\n    if isinstance(value, dict):\n        value = list(value.values())\n    if not isinstance(value, list) or len(value) != 18 or set(value) != set(LABELS):\n        raise ValueError(\'Checkpoint label set differs from the 18 official labels\')\n    if ordered is not None and value != ordered:\n        raise ValueError(\'Checkpoint label order differs from its frozen source head\')\n    return value\n\n\ndef safetensor_header(path):\n    path = Path(path)\n    with path.open(\'rb\') as f:\n        raw = f.read(8)\n        if len(raw) != 8:\n            raise ValueError(\'Truncated safetensors length prefix\')\n        size = struct.unpack(\'<Q\', raw)[0]\n        if size < 2 or size > 16 * 1024 * 1024 or size + 8 > path.stat().st_size:\n            raise ValueError(\'Invalid safetensors header length\')\n        header = json.loads(f.read(size))\n    result = {}\n    for key, value in header.items():\n        if key == \'__metadata__\':\n            continue\n        shape, offsets = value[\'shape\'], value[\'data_offsets\']\n        if (not isinstance(shape, list) or not all(isinstance(x, int) and x >= 0 for x in shape)\n                or not isinstance(offsets, list) or len(offsets) != 2 or offsets[0] < 0\n                or offsets[1] < offsets[0] or offsets[1] > path.stat().st_size - size - 8):\n            raise ValueError(\'Invalid safetensors tensor extent\')\n        result[key] = {\'shape\': shape, \'dtype\': value[\'dtype\'], \'data_offsets\': offsets}\n    if not result:\n        raise ValueError(\'Empty safetensors tensor inventory\')\n    return result\n\n\ndef npz_inventory(path, expected_classes=None, expected_layers=None, hidden=None):\n    result = {}\n    with np.load(path, allow_pickle=False) as data:\n        for key in data.files:\n            try:\n                value = data[key]\n            except ValueError as error:\n                raise ValueError(\'Object/pickle NPZ metadata is not admitted by this CPU audit: \' + key) from error\n            item = {\'shape\': list(value.shape), \'dtype\': str(value.dtype)}\n            if value.dtype.kind in \'fciu\':\n                if not np.isfinite(value).all():\n                    raise ValueError(\'Nonfinite numeric NPZ member: \' + key)\n                item[\'finite\'] = True\n            result[key] = item\n        if expected_classes is not None:\n            checked_labels(data[\'classes\'].tolist(), expected_classes)\n        if expected_layers is not None and data[\'pool_layers\'].tolist() != expected_layers:\n            raise ValueError(\'LoRA pooled layers differ from frozen source\')\n        if hidden is not None and expected_layers is not None:\n            dimension = hidden * len(expected_layers)\n            for key, shape in [(\'plain_coef\', (18, dimension)), (\'plain_intercept\', (18,)),\n                               (\'center_coef\', (18, dimension)), (\'center_intercept\', (18,))]:\n                if key not in data or data[key].shape != shape:\n                    raise ValueError(\'Required LoRA readout missing or dimension mismatch: \' + key)\n            if \'claude_coef\' in data:\n                if data[\'claude_coef\'].shape != (5, dimension) or data[\'claude_intercept\'].shape != (5,):\n                    raise ValueError(\'Optional Claude gate dimensions mismatch\')\n    return result\n\n\ndef validate_recovery(manifest_path):\n    manifest_path = Path(manifest_path)\n    doc = read_json(manifest_path)\n    if doc.get(\'independently_verified_published_migration_manifest\') is not True:\n        raise ValueError(\'Recovery manifest does not declare published migration verification\')\n    records = doc.get(\'files\')\n    if not isinstance(records, dict) or not records:\n        raise ValueError(\'Recovery archive has no asset records\')\n    verified = {}\n    for relative, record in records.items():\n        safe_relative(relative)\n        flat = safe_relative(record[\'flat_file\'])\n        origin = manifest_path.parent / flat\n        if not origin.resolve().is_relative_to(manifest_path.parent.resolve()) or not origin.is_file():\n            raise ValueError(\'Missing/escaping recovery archive asset\')\n        expected = record[\'sha256\']\n        if not isinstance(expected, str) or not re.fullmatch(\'[0-9a-f]{64}\', expected) or sha(origin) != expected:\n            raise ValueError(\'Recovery archive asset SHA mismatch: \' + str(relative))\n        verified[str(relative)] = {\'flat_file\': str(flat), \'bytes\': origin.stat().st_size, \'sha256\': expected}\n    required = {\'COMPLETE.json\', \'readouts.npz\', \'heads.safetensors\',\n                \'adapter/adapter_config.json\', \'adapter/adapter_model.safetensors\', \'tokenizer/tokenizer_config.json\'}\n    if not required.issubset(verified):\n        raise ValueError(\'Recovery archive omits mandatory lora24/final assets: \' + \',\'.join(sorted(required - set(verified))))\n    readout_path = manifest_path.parent / verified[\'readouts.npz\'][\'flat_file\']\n    readout = npz_inventory(readout_path, SORTED_LABELS, [2, 4, 6, 8, 12, 16, 20, 24], hidden=4096)\n    head_path = manifest_path.parent / verified[\'heads.safetensors\'][\'flat_file\']\n    heads = safetensor_header(head_path)\n    return {\'manifest_sha256\': sha(manifest_path), \'verified_files\': verified,\n            \'readouts\': readout, \'head_tensor_headers\': heads,\n            \'provenance_scope\': \'Published manifest declaration and actual byte parity; independent original training provenance is not authenticated here.\'}\n\n\nclass Audit:\n    def __init__(self, root, out):\n        self.root, self.out = Path(root), Path(out)\n        self.checks, self.warnings, self.details, self.code = [], [], {}, []\n\n    def check(self, name, fn):\n        try:\n            value = fn()\n            self.details[name] = value\n            self.checks.append({\'name\': name, \'status\': \'PASS\'})\n            print(\'PREFLIGHT_PASS\', name, flush=True)\n            return value\n        except Exception as error:\n            self.checks.append({\'name\': name, \'status\': \'FAIL\', \'error_type\': type(error).__name__, \'error\': str(error)[:1200]})\n            print(\'PREFLIGHT_FAIL\', name, type(error).__name__, str(error)[:400], flush=True)\n            return None\n\n    def matches(self, pattern):\n        return [Path(p) for p in sorted(glob.glob(str(self.root / pattern), recursive=True))]\n\n    def unique(self, pattern):\n        found = self.matches(pattern)\n        if len(found) != 1:\n            raise ValueError(f\'Required asset {pattern}: expected 1, found {len(found)}\')\n        return found[0]\n\n    def small_code(self, path, role):\n        path = Path(path)\n        if path.stat().st_size > 256 * 1024:\n            raise ValueError(\'Custom code/config exceeds private text export bound: \' + path.name)\n        content = path.read_text(encoding=\'utf-8\')\n        if path.suffix == \'.py\':\n            ast.parse(content)\n        digest = sha(path)\n        record = {\'path\': str(path.relative_to(self.root)), \'role\': role, \'bytes\': path.stat().st_size, \'sha256\': digest}\n        identical = next((item for item in self.code if item[\'sha256\'] == digest and \'text\' in item), None)\n        if identical is None:\n            record[\'text\'] = content\n        else:\n            record[\'same_text_as\'] = identical[\'path\']\n        self.code.append(record)\n        return {\'path\': str(path.relative_to(self.root)), \'bytes\': path.stat().st_size, \'sha256\': digest, \'executed\': False}\n\n    def tokenizer_inventory(self, directory):\n        directory = Path(directory)\n        assets = [p for p in directory.iterdir() if p.is_file() and p.name in\n                  [\'tokenizer.json\', \'tokenizer.model\', \'spm.model\', \'vocab.txt\', \'vocab.json\', \'merges.txt\',\n                   \'tokenizer_config.json\', \'special_tokens_map.json\', \'added_tokens.json\']]\n        if not any(p.name in [\'tokenizer.json\', \'tokenizer.model\', \'spm.model\', \'vocab.txt\', \'vocab.json\'] and p.stat().st_size > 0 for p in assets):\n            raise ValueError(\'Required local tokenizer vocabulary/model is missing\')\n        return [{\'name\': p.name, \'bytes\': p.stat().st_size} for p in sorted(assets)]\n\n    def model_inventory(self, directory):\n        directory = Path(directory)\n        configs = read_json(directory / \'config.json\')\n        files = sorted(p for p in directory.rglob(\'*\') if p.is_file() and \'.cache\' not in p.parts)\n        weights = [p for p in files if p.name.endswith((\'.safetensors\', \'.bin\'))]\n        if not weights:\n            raise ValueError(\'Base/fold checkpoint has no attached weight files\')\n        for index in [p for p in files if p.name.endswith(\'.index.json\')]:\n            doc = read_json(index)\n            for relative in set(doc.get(\'weight_map\', {}).values()):\n                candidate = directory / safe_relative(relative)\n                if not candidate.is_file():\n                    raise ValueError(\'Missing model index shard\')\n        return {\'path\': str(directory.relative_to(self.root)), \'model_type\': configs.get(\'model_type\'),\n                \'hidden_size\': configs.get(\'hidden_size\'), \'num_hidden_layers\': configs.get(\'num_hidden_layers\'),\n                \'config_sha256\': sha(directory / \'config.json\'),\n                \'weight_inventory\': [{\'name\': str(p.relative_to(directory)), \'bytes\': p.stat().st_size} for p in weights],\n                \'weight_values_or_custom_code_executed\': False,\n                \'all_large_weight_bytes_hashed\': False}\n\n    def base(self, name, spec=None, directory_name=None, path_parts=None):\n        matches = []\n        for path in self.matches(\'**/config.json\'):\n            if directory_name is not None and path.parent.name != directory_name:\n                continue\n            if path_parts is not None and not all(part in str(path) for part in path_parts):\n                continue\n            try:\n                cfg = read_json(path)\n            except Exception:\n                continue\n            if spec is None or all(cfg.get(k) == v for k, v in spec.items()):\n                matches.append(path)\n        resolution = {\'selector\': {\'spec\': spec, \'directory_name\': directory_name, \'path_parts\': path_parts},\n                      \'matching_paths_sorted\': [str(p.relative_to(self.root)) for p in matches],\n                      \'source_resolution\': \'first sorted match (ATLAS explicit selector requires exactly one)\',\n                      \'source_would_choose\': str(matches[0].relative_to(self.root)) if matches else None}\n        self.details[name + \'_resolution\'] = resolution\n        if len(matches) != 1:\n            raise ValueError(f\'{name}: missing/ambiguous base selector; {len(matches)} candidates; source first-match resolution recorded for review\')\n        result = self.model_inventory(matches[0].parent)\n        result[\'tokenizer_assets\'] = self.tokenizer_inventory(matches[0].parent)\n        return result\n\n    def main_lora(self, name, hidden, keep):\n        meta_path = self.unique(\'**/\' + name + \'/meta.json\')\n        directory, meta = meta_path.parent, read_json(meta_path)\n        checked_labels(meta[\'labels\'], LABELS)\n        cfg = meta[\'cfg\']; layers = cfg[\'layers\']\n        if cfg[\'keep\'] != keep or cfg.get(\'eos_id\') != 151643 or not (1 <= cfg[\'max_len\'] <= 4096):\n            raise ValueError(\'Main LoRA depth/EOS/token contract differs\')\n        if not layers or not all(isinstance(v, int) and 0 < v <= keep for v in layers):\n            raise ValueError(\'Main LoRA pooling layers invalid\')\n        adapter = directory / \'adapter/adapter_model.safetensors\'\n        acfg = read_json(directory / \'adapter/adapter_config.json\')\n        if acfg.get(\'r\', 0) <= 0 or acfg.get(\'lora_alpha\', 0) <= 0:\n            raise ValueError(\'Main adapter rank/scale invalid\')\n        heads = safetensor_header(directory / \'heads.safetensors\')\n        for prefix in [\'plain\', \'centered\']:\n            if heads[prefix + \'.weight\'][\'shape\'] != [18, hidden * len(layers)] or heads[prefix + \'.bias\'][\'shape\'] != [18]:\n                raise ValueError(\'Main LoRA readout tensor shape differs\')\n        return {\'path\': str(directory.relative_to(self.root)), \'meta_sha256\': sha(meta_path), \'cfg\': cfg,\n                \'adapter_sha256\': sha(adapter), \'adapter_headers\': safetensor_header(adapter), \'head_headers\': heads}\n\n    def component_lora(self, pattern, layers):\n        marker = self.unique(pattern)\n        directory = marker.parent\n        cfg = read_json(directory / \'adapter/adapter_config.json\')\n        header = safetensor_header(directory / \'adapter/adapter_model.safetensors\')\n        norm = directory / (\'norms.safetensors\' if (directory / \'norms.safetensors\').exists() else \'heads.safetensors\')\n        if not (directory / \'tokenizer/tokenizer_config.json\').is_file():\n            raise ValueError(\'Component adapter tokenizer missing\')\n        hidden = 3584 if layers[-1] == 20 else 4096\n        return {\'path\': str(directory.relative_to(self.root)), \'complete_sha256\': sha(marker),\n                \'adapter_sha256\': sha(directory / \'adapter/adapter_model.safetensors\'), \'adapter_cfg\': cfg,\n                \'adapter_header_tensor_count\': len(header), \'norm_headers\': safetensor_header(norm),\n                \'tokenizer_assets\': self.tokenizer_inventory(directory / \'tokenizer\'),\n                \'readouts\': npz_inventory(directory / \'readouts.npz\', SORTED_LABELS, layers, hidden=hidden)}\n\n    def bank(self):\n        bank = self.unique(\'**/synthetic4404.joblib\')\n        receipt = read_json(bank.parent / \'COMPLETE.json\')\n        if (not receipt.get(\'full_fit\') or receipt.get(\'fold_adapter\') or receipt.get(\'adapter_training\')\n                or receipt.get(\'raw_synthetic_data_in_export\')):\n            raise ValueError(\'Night24 bank declared fit/export contract differs\')\n        rec = receipt[\'banks\'][\'synthetic4404\']\n        bank_sha = sha(bank)\n        if rec[\'official_rows\'] != 2699 or rec[\'synthetic_rows\'] != 4404 or bank_sha != rec[\'readout_sha256\']:\n            raise ValueError(\'Night24 bank row count or actual SHA differs\')\n        implementation_sha = sha(bank.parent / \'readouts.py\')\n        declared_code_hashes = {k: v for k, v in receipt.items() if isinstance(v, str) and\n                                \'readout\' in k.lower() and (\'py\' in k.lower() or \'code\' in k.lower()) and \'sha\' in k.lower()}\n        for expected in declared_code_hashes.values():\n            if expected != implementation_sha:\n                raise ValueError(\'Night24 readouts.py differs from its declared implementation SHA\')\n        if not declared_code_hashes:\n            self.warnings.append(\'Night24 COMPLETE does not directly declare a readouts.py implementation SHA; observed bytes are exported and hash-pinned for separate static review.\')\n        return {\'path\': str(bank.relative_to(self.root)), \'bytes\': bank.stat().st_size, \'sha256\': bank_sha,\n                \'complete_sha256\': sha(bank.parent / \'COMPLETE.json\'), \'official_rows\': 2699, \'synthetic_rows\': 4404,\n                \'readouts_python\': self.small_code(bank.parent / \'readouts.py\', \'Night24 custom readout implementation\'),\n                \'declared_readouts_code_hashes\': declared_code_hashes,\n                \'joblib_deserialized\': False,\n                \'bank_internal_adapter_sha_alignment\': \'Not inspected without deserializing; original runtime must verify adapter hash, label order and pool layers.\'}\n\n    def pools(self):\n        pool_path = self.unique(\'**/pool_v4.parquet\')\n        pool = pd.read_parquet(pool_path)\n        if not {\'text\', \'model\'}.issubset(pool.columns) or len(pool) == 0:\n            raise ValueError(\'Main SVC pool missing schema/rows\')\n        if pool.text.isna().any() or not pool.text.map(lambda x: isinstance(x, str) and bool(x.strip())).all():\n            raise ValueError(\'Main SVC pool has empty/non-string text\')\n        if not pool.model.isin(LABELS).all() or set(pool.model) != set(LABELS):\n            raise ValueError(\'Main SVC pool label set differs\')\n        parts, records = [], []\n        for pattern, label_column, finish_column in EXTRA_SOURCES:\n            for path in self.matches(pattern):\n                frame = pd.read_csv(path)\n                if not {\'prompt_id\', label_column, \'text\'}.issubset(frame.columns):\n                    raise ValueError(\'Required extra SVC CSV schema mismatch: \' + str(path.relative_to(self.root)))\n                rows = len(frame)\n                if finish_column and finish_column in frame:\n                    frame = frame[frame[finish_column].eq(\'stop\')]\n                frame = frame.rename(columns={label_column: \'model\'})[[\'prompt_id\', \'model\', \'text\']].copy()\n                frame[\'prompt_id\'] = (\'pv_\' if path.name == \'passages_clean.csv\' else \'\') + frame.prompt_id.astype(str)\n                parts.append(frame)\n                records.append({\'path\': str(path.relative_to(self.root)), \'sha256\': sha(path), \'rows_before_source_finish_filter\': rows,\n                                \'rows_after_source_finish_filter\': len(frame), \'source_finish_column\': finish_column})\n        if not parts:\n            raise ValueError(\'No external SVC passage sources resolved; source would silently train a different no-extra variant\')\n        extra = pd.concat(parts, ignore_index=True)\n        extra = extra[extra.model.isin(LABELS)].copy()\n        extra[\'text\'] = extra.text.astype(str).map(lambda t: re.sub(r\'\\s+\', \' \', re.sub(r\'^\\*\\*(.*)\\*\\*$\', r\'\\1\', t.strip())).strip())\n        extra = extra[extra.text.str.split().str.len().between(30, 300)].drop_duplicates([\'model\', \'text\'])\n        extra = extra.drop_duplicates([\'prompt_id\', \'model\'])\n        extra = extra[extra.groupby(\'prompt_id\').model.transform(\'nunique\') >= 12]\n        if len(extra) == 0 or extra.text.str.strip().eq(\'\').any():\n            raise ValueError(\'Filtered extra SVC pool is empty\')\n        quality = []\n        for path in self.matches(\'**/quality_pool_public.parquet\'):\n            frame = pd.read_parquet(path)\n            valid_schema = {\'text\', \'model\'}.issubset(frame.columns)\n            quality.append({\'path\': str(path.relative_to(self.root)), \'sha256\': sha(path), \'rows\': len(frame),\n                            \'columns\': list(frame.columns), \'nonempty\': len(frame) > 0,\n                            \'text_model_schema\': valid_schema,\n                            \'all_labels_allowed\': bool(frame.model.isin(LABELS).all()) if valid_schema else False,\n                            \'all_text_nonempty_strings\': bool(frame.text.map(lambda x: isinstance(x, str) and bool(x.strip())).all()) if valid_schema else False,\n                            \'direct_outer_or_comp_stack_reference\': False,\n                            \'use_scope\': \'Attachment inventory only; no direct reference in frozen outer/comp source. Separate readouts.py review must determine any indirect helper use.\'})\n        return {\'main_pool\': {\'path\': str(pool_path.relative_to(self.root)), \'sha256\': sha(pool_path), \'rows\': len(pool),\n                              \'columns\': list(pool.columns), \'label_counts\': pool.model.value_counts().sort_index().to_dict()},\n                \'extra_csv_sources\': records, \'source_exact_filtered_extra_rows\': len(extra),\n                \'source_exact_filtered_extra_prompts\': int(extra.prompt_id.nunique()),\n                \'source_exact_filtered_extra_label_counts\': extra.model.value_counts().sort_index().to_dict(),\n                \'quality_pool_optional_attachment_inventory\': quality,\n                \'quality_named_assets\': [str(p.relative_to(self.root)) for p in self.matches(\'**/*\')\n                                         if p.is_file() and \'quality\' in str(p).lower() and p.suffix in [\'.csv\', \'.parquet\', \'.json\']],\n                \'provenance_limit\': \'Source-defined final-inference pool reproduced structurally; no blanket provider or prompt-independence certification.\'}\n\n    def ultrabert(self):\n        candidates = []\n        for root in self.matches(\'**/ultrabert-380m-ntl\'):\n            folds = [p for p in sorted(root.glob(\'fold*\')) if (p / \'config.json\').is_file()\n                     and read_json(p / \'config.json\').get(\'model_type\') == \'ultrabert\']\n            if len(folds) == 5:\n                candidates.append((root, folds))\n        if len(candidates) != 1:\n            raise ValueError(\'Missing/ambiguous Ultrabert root containing exactly five valid folds\')\n        root, folds = candidates[0]\n        result = []\n        for fold in folds:\n            cfg = read_json(fold / \'config.json\')\n            checked_labels(cfg[\'id2label\'])\n            exported = [self.small_code(fold / \'config.json\', \'Ultrabert fold config\')]\n            for name, mapping in cfg.get(\'auto_map\', {}).items():\n                for target in mapping if isinstance(mapping, list) else [mapping]:\n                    if target is None:\n                        continue\n                    if \'--\' in target:\n                        raise ValueError(\'Remote-repository Ultrabert auto_map requires explicit static resolution\')\n                    module = target.rsplit(\'.\', 1)[0].replace(\'.\', \'/\') + \'.py\'\n                    if not (fold / safe_relative(module)).is_file():\n                        raise ValueError(\'Ultrabert auto_map local code file missing\')\n            for path in sorted(fold.rglob(\'*.py\')):\n                exported.append(self.small_code(path, \'Ultrabert custom code; never imported by preflight\'))\n            tokenizer = fold / \'tokenizer_config.json\'\n            if tokenizer.is_file():\n                exported.append(self.small_code(tokenizer, \'Ultrabert tokenizer config\'))\n            result.append({\'inventory\': self.model_inventory(fold), \'private_review_exports\': exported})\n        if not any((folds[0] / name).is_file() for name in [\'tokenizer.json\', \'tokenizer.model\', \'spm.model\', \'vocab.txt\', \'vocab.json\']):\n            raise ValueError(\'First Ultrabert fold has no local tokenizer asset\')\n        return {\'root\': str(root.relative_to(self.root)), \'folds\': result, \'trust_remote_code_executed\': False}\n\n    def atlas(self):\n        path = self.unique(\'**/atlas_bundle.json\'); directory = path.parent\n        cfg = read_json(path)\n        if not isinstance(cfg.get(\'training_config\'), dict) or cfg[\'training_config\'].get(\'max_length\', 0) <= 0:\n            raise ValueError(\'ATLAS training/token-length config missing\')\n        required = [\'ntl_atlas/model.py\', \'ntl_rlcd/data.py\']\n        exports = [self.small_code(path, \'ATLAS bundle config\')]\n        for relative in required:\n            if not (directory / relative).is_file():\n                raise ValueError(\'ATLAS local custom package asset missing: \' + relative)\n        for package in [\'ntl_atlas\', \'ntl_rlcd\']:\n            for code in sorted((directory / package).rglob(\'*.py\')):\n                exports.append(self.small_code(code, \'ATLAS local code; never imported by preflight\'))\n        return {\'path\': str(directory.relative_to(self.root)), \'config_sha256\': sha(path),\n                \'adapter_sha256\': sha(directory / \'adapters.safetensors\'),\n                \'adapter_headers\': safetensor_header(directory / \'adapters.safetensors\'),\n                \'private_review_exports\': exports}\n\n    def execute(self):\n        official_candidates = [self.root / \'competitions/name-that-llm\', self.root / \'name-that-llm\']\n        official = next((p for p in official_candidates if (p / \'test.csv\').is_file()), None)\n        def official_contract():\n            if official is None or not (official / \'train.csv\').is_file():\n                raise ValueError(\'Official competition train/test path unavailable\')\n            if sha(official / \'train.csv\') != TRAIN_SHA:\n                raise ValueError(\'Official training hash changed\')\n            frame = pd.read_csv(official / \'train.csv\', dtype=str, keep_default_na=False)\n            if len(frame) != 2699 or frame.prompt_id.nunique() != 150 or set(frame.model) != set(LABELS):\n                raise ValueError(\'Official dimensions/labels changed\')\n            return {\'train_sha256\': TRAIN_SHA, \'rows\': len(frame), \'prompts\': 150, \'classes\': 18,\n                    \'test_file_opened_or_hashed\': False,\n                    \'preview_warning\': \'Frozen exact-three-row preview bypasses ALL model/input paths and emits constant labels; CPU pass is not GPU inference evidence.\'}\n        self.check(\'official_train\', official_contract)\n        self.check(\'legacy_recovery_archive\', lambda: validate_recovery(self.unique(\'**/legacy_recovery_manifest.json\')))\n        self.check(\'night24_synthetic4404_bank\', self.bank)\n        self.check(\'main_and_quality_selected_extra_svc_pools\', self.pools)\n        for name, spec in [\n            (\'comp_glm9b\', {\'model_type\': \'glm4\', \'hidden_size\': 4096}),\n            (\'comp_qwen3_8b\', {\'model_type\': \'qwen3\', \'hidden_size\': 4096}),\n            (\'comp_qwen25_7b\', {\'model_type\': \'qwen2\', \'hidden_size\': 3584}),\n            (\'comp_qwen3_4b\', {\'model_type\': \'qwen3\', \'hidden_size\': 2560}),\n            (\'comp_token_attention_qwen35\', {\'model_type\': \'qwen3_5\'}),\n            (\'main_qwen3_14b\', {\'model_type\': \'qwen3\', \'hidden_size\': 5120, \'num_hidden_layers\': 40}),\n            (\'main_qwen25_14b\', {\'model_type\': \'qwen2\', \'hidden_size\': 5120, \'num_hidden_layers\': 48}),\n        ]:\n            self.check(name, lambda name=name, spec=spec: self.base(name, spec))\n        for directory in [\'deberta-v3-large\', \'electra-large-discriminator\', \'desklib-detector\']:\n            self.check(\'comp_encoder_\' + directory, lambda directory=directory: self.base(\'comp_encoder_\' + directory, directory_name=directory))\n        self.check(\'atlas_qwen3_8b_base\', lambda: self.base(\'atlas_qwen3_8b_base\', path_parts=[\'/qwen-3/\', \'/8b-base/\']))\n        self.check(\'main_lora_qwen3_14b\', lambda: self.main_lora(\'lora_qwen3_14b\', 5120, 27))\n        self.check(\'main_lora_qwen25_14b\', lambda: self.main_lora(\'lora_qwen25_14b\', 5120, 32))\n        for name, pattern, layers in [\n            (\'comp_lora24_seed7\', \'**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json\', [2,4,6,8,12,16,20,24]),\n            (\'comp_lora24_claude\', \'**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json\', [2,4,6,8,12,16,20,24]),\n            (\'comp_lora20_q25\', \'**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json\', [2,4,6,8,10,12,16,20]),\n        ]:\n            self.check(name, lambda pattern=pattern, layers=layers: self.component_lora(pattern, layers))\n        def main_deberta():\n            path = self.unique(\'**/deberta_v3_large/meta.json\'); meta = read_json(path)\n            checked_labels(meta[\'labels\'], LABELS)\n            if not 1 <= meta.get(\'max_len\', 0) <= 4096:\n                raise ValueError(\'Main DeBERTa token length metadata invalid\')\n            inventory = self.model_inventory(path.parent / \'encoder\')\n            heads = safetensor_header(path.parent / \'head.safetensors\')\n            if heads[\'weight\'][\'shape\'] != [18, inventory[\'hidden_size\']] or heads[\'bias\'][\'shape\'] != [18]:\n                raise ValueError(\'Main DeBERTa head dimensions differ\')\n            return {\'meta_sha256\': sha(path), \'meta_max_len\': meta[\'max_len\'],\n                    \'encoder\': inventory,\n                    \'tokenizer_assets\': self.tokenizer_inventory(path.parent / \'encoder\'),\n                    \'head_headers\': heads}\n        self.check(\'main_adapted_deberta\', main_deberta)\n        for filename in [\'tokattn_final.pt\', \'tokattn_debl23_final.pt\']:\n            def pt_asset(filename=filename):\n                path = self.unique(\'**/\' + filename)\n                if path.stat().st_size == 0:\n                    raise ValueError(\'Empty Torch head file\')\n                return {\'path\': str(path.relative_to(self.root)), \'bytes\': path.stat().st_size,\n                        \'sha256\': sha(path), \'torch_or_pickle_deserialized\': False}\n            self.check(filename, pt_asset)\n        self.check(\'ultrabert_five_folds\', self.ultrabert)\n        self.check(\'atlas_bundle_and_custom_packages\', self.atlas)\n        failed = [c for c in self.checks if c[\'status\'] != \'PASS\']\n        self.out.mkdir(parents=True, exist_ok=True)\n        code_export = {\'scope\': \'PRIVATE static text for review; code has not been executed\', \'files\': self.code}\n        (self.out / \'PRIVATE_CODE_EXPORT.json\').write_text(json.dumps(code_export, indent=2) + \'\\n\', encoding=\'utf-8\')\n        report = {\'stage\': \'10\', \'source_version\': SOURCE_VERSION, \'upstream_source_sha256\': UPSTREAM_SHA,\n                  \'embedded_complement_source_sha256\': COMP_STACK_SHA,\n                  \'status\': \'PASS_CPU_INPUT_CONTRACT_ONLY\' if not failed else \'BLOCKED_INPUT_CONTRACT\',\n                  \'passed\': len(self.checks) - len(failed), \'total\': len(self.checks), \'checks\': self.checks,\n                  \'details\': self.details, \'warnings\': self.warnings, \'audit_script_sha256\': sha(Path(__file__)),\n                  \'private_code_export_sha256\': sha(self.out / \'PRIVATE_CODE_EXPORT.json\'),\n                  \'private_code_export_files\': len(self.code), \'model_or_custom_code_executed\': False,\n                  \'joblib_pickle_torch_checkpoint_deserialized\': False, \'test_labels_or_hidden_text_accessed\': False,\n                  \'training_or_inference_started\': False,\n                  \'limitations\': [\'CPU inventory/metadata/hash checks cannot establish GPU model parity, finite model outputs or runtime budget.\',\n                                 \'Safetensors headers prove tensor layout only, not finite tensor values.\',\n                                 \'The Night24 joblib internals are not deserialized; adapter/label/layer compatibility must pass the original runtime assertions.\',\n                                 \'The source normal preview emits constants without loading weights; never report it as a real inference preview.\',\n                                 \'Multiple source-first base matches are recorded and blocked for explicit resolution, rather than silently choosing different bytes.\',\n                                 \'The original recipe fits final heads on official rows; its in-sample checks and fitted bank are not honest OOF accuracy.\']}\n        (self.out / \'INPUT_AUDIT.json\').write_text(json.dumps(report, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n        print(\'STAGE10_CPU_AUDIT_SUMMARY\', json.dumps({k: report[k] for k in [\'status\',\'passed\',\'total\',\'private_code_export_files\',\'model_or_custom_code_executed\']}), flush=True)\n        return report\n\n\ndef self_test():\n    checks = {}\n    def rejects(name, fn):\n        try:\n            fn()\n        except Exception:\n            checks[name] = True\n        else:\n            checks[name] = False\n    for value in [\'/outside\', \'../escape\', \'folder/../escape\']:\n        rejects(\'path_traversal_\' + str(len(checks)), lambda value=value: safe_relative(value))\n    rejects(\'label_set_mismatch\', lambda: checked_labels([\'wrong\'] * 18))\n    rejects(\'label_order_mismatch\', lambda: checked_labels(SORTED_LABELS, LABELS))\n    with tempfile.TemporaryDirectory() as td:\n        root = Path(td); a = Audit(root, root / \'out\')\n        rejects(\'missing_recovery_archive\', lambda: a.unique(\'**/legacy_recovery_manifest.json\'))\n        (root / \'one\').mkdir(); (root / \'two\').mkdir()\n        (root / \'one/legacy_recovery_manifest.json\').write_text(\'{}\')\n        (root / \'two/legacy_recovery_manifest.json\').write_text(\'{}\')\n        rejects(\'ambiguous_recovery_archive\', lambda: a.unique(\'**/legacy_recovery_manifest.json\'))\n        payload = root / \'payload\'; payload.write_bytes(b\'fixture\')\n        m = root / \'manifest.json\'\n        m.write_text(json.dumps({\'independently_verified_published_migration_manifest\': True,\n                                 \'files\': {\'COMPLETE.json\': {\'flat_file\':\'payload\',\'sha256\':\'0\'*64}}}))\n        rejects(\'recovery_actual_hash_mismatch\', lambda: validate_recovery(m))\n        m.write_text(json.dumps({\'independently_verified_published_migration_manifest\': True,\n                                 \'files\': {\'../escape\': {\'flat_file\':\'payload\',\'sha256\':sha(payload)}}}))\n        rejects(\'recovery_relative_path_escape\', lambda: validate_recovery(m))\n        invalid = root / \'bad.safetensors\'; invalid.write_bytes(struct.pack(\'<Q\', 1000) + b\'{}\')\n        rejects(\'truncated_safetensors_header\', lambda: safetensor_header(invalid))\n        np.savez(root / \'object.npz\', classes=np.array([\'bad\'], dtype=object))\n        rejects(\'object_npz_pickle_rejected\', lambda: npz_inventory(root / \'object.npz\'))\n        np.savez(root / \'nonfinite.npz\', weight=np.array([np.nan]))\n        rejects(\'nonfinite_numeric_npz_rejected\', lambda: npz_inventory(root / \'nonfinite.npz\'))\n        np.savez(root / \'missing_readout.npz\', classes=np.array(SORTED_LABELS), pool_layers=np.array([2]))\n        rejects(\'missing_required_readout_key_rejected\', lambda: npz_inventory(root / \'missing_readout.npz\', SORTED_LABELS, [2], hidden=1))\n    report = {\'status\': \'PASS\' if all(checks.values()) else \'FAIL\', \'checks\': checks,\n              \'passed\': sum(checks.values()), \'total\': len(checks), \'source_or_model_executed\': False}\n    if not all(checks.values()):\n        raise ValueError(json.dumps(report))\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--root\', type=Path, default=Path(\'/kaggle/input\'))\n    parser.add_argument(\'--out\', type=Path, default=Path(\'/kaggle/working/stage10_preflight\'))\n    parser.add_argument(\'--self-test\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.self_test:\n        result = self_test()\n        args.out.parent.mkdir(parents=True, exist_ok=True)\n        args.out.write_text(json.dumps(result, indent=2) + \'\\n\')\n        print(json.dumps(result, indent=2))\n    else:\n        result = Audit(args.root, args.out).execute()\n        if result[\'status\'] != \'PASS_CPU_INPUT_CONTRACT_ONLY\':\n            raise SystemExit(\'Stage10 input contract blocked; inspect INPUT_AUDIT.json before GPU execution.\')\n'
assert hashlib.sha256(AUDIT_SOURCE.encode()).hexdigest()=='cece257d40cfbfbf2911ebc10c303a85df9e9caa854175f9852326828c5eb970'
p=Path('/kaggle/working/stage10_input_preflight.py');p.write_text(AUDIT_SOURCE)
namespace={'__name__':'stage10_read_only_audit','__file__':str(p)}
exec(compile(AUDIT_SOURCE,str(p),'exec'),namespace)
report=namespace['Audit']('/kaggle/input','/kaggle/working/stage10_preflight').execute()
print('STAGE10_INPUT_AUDIT_JSON_BEGIN')
print(json.dumps(report,indent=2,allow_nan=False))
print('STAGE10_INPUT_AUDIT_JSON_END')
private_export=Path('/kaggle/working/stage10_preflight/PRIVATE_CODE_EXPORT.json').read_bytes()
encoded=base64.b64encode(gzip.compress(private_export,mtime=0)).decode()
chunks=[encoded[i:i+6000] for i in range(0,len(encoded),6000)]
for i,chunk in enumerate(chunks):print('STAGE10_PRIVATE_CODE_EXPORT_B64',i+1,len(chunks),chunk)
print('STAGE10_CPU_AUDIT_FINISHED',report['status'])
